In [1]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. 高速スケールフリー・ネットワーク生成 (Numba JIT)
# ==========================================
@njit
def build_ba_edges_fast(N, m):
    """Barabási-Albertモデルに基づくエッジリストの高速生成"""
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)

    idx = 0
    target_idx = 0

    # 初期完全グラフの構築
    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i
                col[idx] = j
                idx += 1
                targets[target_idx] = i
                target_idx += 1

    chosen = np.empty(m, dtype=np.int32)
    # 優先的選択（Preferential Attachment）
    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True
                    break
            if not already:
                chosen[c_idx] = node
                c_idx += 1

        for k in range(m):
            target = chosen[k]
            # 双方向エッジの追加
            row[idx] = i
            col[idx] = target
            idx += 1
            row[idx] = target
            col[idx] = i
            idx += 1
            # ターゲットリストの更新（次数比例）
            targets[target_idx] = i
            target_idx += 1
            targets[target_idx] = target
            target_idx += 1

    return row[:idx], col[:idx]

# ==========================================
# 2. シミュレーション関数 (情報疫学モデル)
# ==========================================
def run_simulation(N, A_sparse, degrees, source_node, v_target, params):
    dt = params['dt']
    steps = params['steps']
    T_expose = params['T_expose']

    # 乱数による初期値設定（初期値依存の排除）
    # メモリ節約のためfloat32を使用
    X = np.random.uniform(0, 0.2, N).astype(np.float32)
    D = np.random.uniform(0, 0.1, N).astype(np.float32)

    history_D_min = []
    history_D_maj = []

    # マイノリティインデックスの取得
    min_idx = (v_target == 1.0)
    maj_idx = ~min_idx

    # 偽情報の強度
    b_val = 0.9
    b = np.zeros(N, dtype=np.float32)

    for step in tqdm(range(steps), desc=f"Simulating", leave=False):
        # T_exposeまではソースノードがフェイク（ウイルス）を発信し続ける
        if step < T_expose:
            b[source_node] = b_val
        else:
            b[source_node] = 0.0 # フェイクの暴露・BAN

        # 隣接ノードからの局所的重力場（同調圧力）の計算
        # P_i = Σ A_ij X_j / degree_i
        neighbor_sum = A_sparse.dot(X)
        P = neighbor_sum / degrees

        # SDEの更新 (Euler-Maruyama法)
        # 1. サイトカイン（同調度 X）の進行
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X_new = np.clip(X + dX, 0, 1)

        # 2. 組織壊死（分断度 D）と線維化（修復機能の喪失）
        # dD = k_break * v * X * (1-D) - k_restore * (1-X) * D * exp(-γD)
        repair_capacity = np.exp(-params['gamma'] * D)
        dD = (params['k_break'] * v_target * X_new * (1 - D) -
              params['k_restore'] * (1 - X_new) * D * repair_capacity) * dt
        D_new = np.clip(D + dD, 0, 1)

        X = X_new
        D = D_new

        history_D_min.append(np.mean(D[min_idx]))
        history_D_maj.append(np.mean(D[maj_idx]))

    return history_D_min, history_D_maj

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    # ターゲットスケール群
    N_list = [1000, 5000, 10000, 50000, 100000, 1000000, 10000000]
    m = 2 # BAモデルのエッジ追加数

    # SDEパラメータ
    params = {
        'dt': 0.1,
        'steps': 100,
        'T_expose': 50,
        'sigma': 0.05,
        'k_break': 0.5,
        'k_restore': 0.1,
        'gamma': 5.0
    }

    output_dir = "tmfg_hypothesis1_results"
    os.makedirs(output_dir, exist_ok=True)

    summary_data = []

    for N in N_list:
        print(f"\n[{N} Nodes] ネットワーク生成中 (Scale-Free BA Model)...")
        try:
            # ネットワーク生成と疎行列化
            row, col = build_ba_edges_fast(N, m)
            data = np.ones(len(row), dtype=np.float32)
            A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
            del row, col, data
            gc.collect()

            # 孤立ノード回避のため次数は最低1とする
            degrees = np.array(A_sparse.sum(axis=1)).flatten()
            degrees[degrees == 0] = 1.0

            # ハブノード（最大次数）と一般ノード（中央値以下）の特定
            hub_node = np.argmax(degrees)
            median_deg = np.median(degrees)
            rand_node_candidates = np.where(degrees <= median_deg)[0]
            rand_node = np.random.choice(rand_node_candidates)

            # マイノリティの設定（5%をランダムに選出）
            v_target = np.full(N, 0.1, dtype=np.float32) # マジョリティ
            min_count = max(1, int(N * 0.05))
            min_indices = np.random.choice(N, min_count, replace=False)
            v_target[min_indices] = 1.0 # マイノリティ（基礎疾患・免疫的脆弱性）

            print(f"-> シナリオA: ハブノード（スーパースプレッダー）発信の計算")
            hist_min_hub, hist_maj_hub = run_simulation(N, A_sparse, degrees, hub_node, v_target, params)

            print(f"-> シナリオB: 一般ノード発信の計算")
            hist_min_rand, hist_maj_rand = run_simulation(N, A_sparse, degrees, rand_node, v_target, params)

            # メモリ解放
            del A_sparse, degrees, v_target
            gc.collect()

            # CSV出力
            df = pd.DataFrame({
                'TimeStep': np.arange(params['steps']),
                'D_min_Hub': hist_min_hub,
                'D_maj_Hub': hist_maj_hub,
                'D_min_Rand': hist_min_rand,
                'D_maj_Rand': hist_maj_rand
            })
            csv_path = os.path.join(output_dir, f'result_N{N}.csv')
            df.to_csv(csv_path, index=False)

            # 個別グラフの出力
            plt.figure(figsize=(10, 6))
            plt.plot(df['TimeStep'], df['D_min_Hub'], 'r-', linewidth=2.5, label='Minority Defect (Hub Origin)')
            plt.plot(df['TimeStep'], df['D_min_Rand'], 'r--', linewidth=1.5, alpha=0.7, label='Minority Defect (Random Origin)')
            plt.plot(df['TimeStep'], df['D_maj_Hub'], 'b-', linewidth=2.5, label='Majority Defect (Hub Origin)')
            plt.axvline(x=params['T_expose'], color='k', linestyle=':', label='Fake Exposed (b=0)')
            plt.title(f'Topological Defect Dynamics: Hub vs Random Origin (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Topological Defect (D)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            png_path = os.path.join(output_dir, f'plot_N{N}.png')
            plt.savefig(png_path, dpi=300, bbox_inches='tight')
            plt.close()

            # サマリー用のデータ保存（最終ステップのマイノリティ分断度）
            summary_data.append({
                'N': N,
                'Final_D_min_Hub': hist_min_hub[-1],
                'Final_D_min_Rand': hist_min_rand[-1]
            })

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足 (OOM) が発生しました。これ以降の巨大Nはスキップし、現在までのデータでZIP化へ移行します。")
            break

    # ==========================================
    # 4. サマリーグラフとZIP化
    # ==========================================
    if len(summary_data) > 0:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'summary_all_N.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Final_D_min_Hub'], 'r-o', label='Hub Origin (Black Hole Effect)')
        plt.plot(df_sum['N'], df_sum['Final_D_min_Rand'], 'g--s', label='Random Origin')
        plt.xscale('log')
        plt.title('Asymptotic Bias and Hub Impact across Macro-Scales')
        plt.xlabel('Sample Size N (Log Scale)')
        plt.ylabel('Final Minority Defect (Hysteresis)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()

        # ZIP化処理
        zip_filename = 'TMFG_Hypothesis1_Results.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)

        print(f"\n🎉 全プロセスの完了: 計算結果とグラフを '{zip_filename}' に出力しました！")

if __name__ == "__main__":
    main()


[1000 Nodes] ネットワーク生成中 (Scale-Free BA Model)...
-> シナリオA: ハブノード（スーパースプレッダー）発信の計算


-> シナリオB: 一般ノード発信の計算



[5000 Nodes] ネットワーク生成中 (Scale-Free BA Model)...
-> シナリオA: ハブノード（スーパースプレッダー）発信の計算


-> シナリオB: 一般ノード発信の計算



[10000 Nodes] ネットワーク生成中 (Scale-Free BA Model)...
-> シナリオA: ハブノード（スーパースプレッダー）発信の計算


...


[1000000 Nodes] ネットワーク生成中 (Scale-Free BA Model)...
-> シナリオA: ハブノード（スーパースプレッダー）発信の計算


-> シナリオB: 一般ノード発信の計算



[10000000 Nodes] ネットワーク生成中 (Scale-Free BA Model)...
-> シナリオA: ハブノード（スーパースプレッダー）発信の計算


-> シナリオB: 一般ノード発信の計算



🎉 全プロセスの完了: 計算結果とグラフを 'TMFG_Hypothesis1_Results.zip' に出力しました！
